# 0.0. Setup

In [49]:
import os
import gc
import copy
import json
import requests
from pathlib import Path

import re
import pickle
import importlib
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
DATA_PUBLIC_DIR = PROJECT_DIR / 'data/public'
DATA_PRIVATE_DIR = PROJECT_DIR / 'data/private'
DATA_PROCESSED_DIR = PROJECT_DIR / 'data/processed'
OUTPUT_DIR = PROJECT_DIR / 'output'

# 1.0. Load data

In [2]:
!ls '{DATA_PRIVATE_DIR}'

'2024-2025 GASTPE Participating Schools Top-up Data.xlsx'
 Alphalist-Schools-Slots-addon_slots.csv
 Alphalist-Schools-Slots-fixed_slots.csv
 Alphalist-Schools-Slots-incentive_slots.csv
 ESC
'ESC and SHSVP Tuition.xlsx'
'GASTPE Yearend Report SY 24-25 (Certfication Annexes).pdf'
'Private School Seats and TOSF ao 2025Oct27.xlsx'
 SHSVP
'SUMMARY OF PRIVATE SCHOOL PROFILE.xlsx'
'SY 23-24 ESC Slots as of 020426.xlsx'
 gastpe_certification_annexes_sy2425.csv
 priv_classroom_furniture.xlsx
 raw_validation_sheets
'~$SY 23-24 ESC Slots as of 020426.xlsx'


In [3]:
fnames = [fn for fn in os.listdir(DATA_PRIVATE_DIR) if re.search(r'alphalist-schools', fn, flags=re.IGNORECASE)]
fpaths = [os.path.join(DATA_PRIVATE_DIR, fn) for fn in fnames]
display(fpaths)

['/workspace/project_paaral/data/private/Alphalist-Schools-Slots-addon_slots.csv',
 '/workspace/project_paaral/data/private/Alphalist-Schools-Slots-fixed_slots.csv',
 '/workspace/project_paaral/data/private/Alphalist-Schools-Slots-incentive_slots.csv']

## 1.1. Read all

In [4]:
dfs = []
for fpath in fpaths:
    df = pd.read_csv(fpath)
    dfs.append(df)

## 1.2. Inspect

In [5]:
print(dfs[0].shape)
display(dfs[0].head(2))
display(dfs[0].tail(2))

(4550, 3)


,School ID,School Name,Value
0,303929,"One La Salle Educational Foundation, Inc.",0
1,1104375,Newfoundland Arts and Science Academy of Tagum...,0


,School ID,School Name,Value
4548,1704003,"DR. GREGORIO VALDEZ INSTITUTE, INC.",0
4549,1704253,Southwestern Institute of Business and Technol...,0


In [6]:
print(dfs[1].shape)
display(dfs[1].head(2))
display(dfs[1].tail(2))

(4550, 3)


,School ID,School Name,Value
0,303929,"One La Salle Educational Foundation, Inc.",38
1,1104375,Newfoundland Arts and Science Academy of Tagum...,50


,School ID,School Name,Value
4548,1704003,"DR. GREGORIO VALDEZ INSTITUTE, INC.",11
4549,1704253,Southwestern Institute of Business and Technol...,50


In [7]:
print(dfs[2].shape)
display(dfs[2].head(2))
display(dfs[2].tail(2))

(4550, 3)


,School ID,School Name,Value
0,303929,"One La Salle Educational Foundation, Inc.",0
1,1104375,Newfoundland Arts and Science Academy of Tagum...,0


,School ID,School Name,Value
4548,1704003,"DR. GREGORIO VALDEZ INSTITUTE, INC.",0
4549,1704253,Southwestern Institute of Business and Technol...,0


# 2.0. Process

## 2.1. Aggregate slots

In [8]:
dfs_tmp = copy.copy(dfs)

csv_label = [
    'addon',
    'fixed',
    'incentive'
]

dfs_processed = []
for df, label in zip(dfs_tmp, csv_label):
    df['School ID'] = df['School ID'].astype(str)
    df['Value'] = df['Value'].astype(int)
    
    new_columns = ['esc_school_id','school_name','count']
    df.columns = new_columns
    df.rename(
        columns={
            'count':'count_'+label+'_slots'
        }, inplace=True
    )
    df.set_index(df.columns[0], inplace=True)
    df.drop(columns='school_name', inplace=True)

    dfs_processed.append(df)

dfs_processed = pd.concat(dfs_processed, axis=1)
dfs_processed['total_count_slots'] = dfs_processed.sum(axis=1)

print(dfs_processed.shape)

(4550, 4)


In [9]:
dfs_processed

,count_addon_slots,count_fixed_slots,count_incentive_slots,total_count_slots
esc_school_id,,,,
303929,0,38,0,38
1104375,0,50,0,50
1504390,44,50,0,94
304395,0,50,0,50
1404408,0,50,0,50
...,...,...,...,...
1704000,0,0,0,0
1704001,0,12,0,12
1704002,10,50,0,60


## 2.2. Match ESC IDs

In [10]:
# We'll use the GASTPE dataset to make a DepEd-PEAC school id match
fpath = os.path.join(DATA_PRIVATE_DIR, 'ESC and SHSVP Tuition.xlsx')
gastpe_excel = pd.read_excel(fpath, sheet_name='Tuition', engine='calamine')
print(gastpe_excel.shape)

(5188, 15)


In [11]:
display(gastpe_excel.head(2))

,Region SHS,Program,SUC/LUC,Billed in SY 2024-2025 (ESC),ESC School ID,DepEd School Id,School Name,ESC (Tuition),ESC (Other),ESC (Misc),ESC (Total),SHSVP (Tuition),SHSVP (Other),SHSVP (Misc),SHSVP (Total)
0,Region I,BOTH,NaN,NaN,100008.0,400001,"St. Andrew Academy of Bacarra, Inc.",13353.05,3817.85,2220.0,19390.90,15515.0,3210.0,2220.0,20945.0
1,Region I,BOTH,NaN,NaN,100018.0,400002,"Badoc Junior College, Inc.",9808.26,908.98,0.0,10717.24,15000.0,2658.0,0.0,17658.0


### Process

In [12]:
essential_columns = ['DepEd School Id','ESC School ID']
df_tmp = gastpe_excel[essential_columns].copy()

new_columns = ['school_id','esc_school_id']
df_tmp.columns = new_columns

# ESC School ID columns has NaNs that is why its float
for col in df_tmp.columns:
    df_tmp[col] = df_tmp[col].astype(str)

mask_na = df_tmp['esc_school_id'].str.contains(r'nan', flags=re.IGNORECASE)
df_tmp.loc[~mask_na, 'esc_school_id'] = df_tmp.loc[~mask_na, 'esc_school_id'].apply(lambda x: re.findall(r'(.*).0', x)[0])
df_tmp.loc[mask_na, 'esc_school_id'] = 'No ESC School ID'

print(df_tmp.shape)

# Create dictionary of ESC-DepEd school ID pairs
deped_esc_ids = {k:v for k,v in zip(df_tmp['school_id'], df_tmp['esc_school_id'])}
esc_deped_ids = {v:k for k,v in deped_esc_ids.items()}

(5188, 2)


In [48]:
# Uncomment to inspect
display(deped_esc_ids)

{'400001': '100008',
 '400002': '100018',
 '400003': '100023',
 '400004': '100025',
 '400006': '100013',
 '400007': '100020',
 '400008': '100021',
 '400010': '100024',
 '400011': '103753',
 '400012': '100028',
 '400013': '100019',
 '400015': '100012',
 '400016': '100026',
 '400017': '100006',
 '400018': '100010',
 '400019': '100007',
 '400020': '100009',
 '400021': '100027',
 '400022': '100029',
 '400023': '100017',
 '400025': '100015',
 '400026': '100016',
 '400027': '100034',
 '400028': '100040',
 '400029': '100031',
 '400031': '100047',
 '400032': '100045',
 '400035': '100039',
 '400036': '100038',
 '400037': '100035',
 '400039': '100042',
 '400040': '100057',
 '400043': '100036',
 '400044': '100052',
 '400045': '100037',
 '400046': '100043',
 '400047': '100044',
 '400048': '100050',
 '400049': '100051',
 '400050': '100053',
 '400051': '100054',
 '400052': '100048',
 '400053': '100055',
 '400054': '100056',
 '400055': '100030',
 '400056': '100046',
 '400061': '100032',
 '400062': '1

### Match

In [14]:
dfs_processed.index.map(esc_deped_ids)

Index(['418013', '408365', '406245', '419005',      nan, '404601', '446545',
       '401884', '406004', '400609',
       ...
       '403425', '403514', '403533', '430502', '430016',      nan, '408013',
       '403391', '407742', '406959'],
      dtype='object', name='esc_school_id', length=4550)

In [15]:
df_match = dfs_processed.copy()
df_match['school_id'] = df_match.index.map(esc_deped_ids)
df_match['school_id'] = df_match['school_id'].astype(str)

df_match['has_deped_school_id'] = None
mask = df_match['school_id'] == 'nan'
df_match.loc[mask, 'has_deped_school_id'] = False
df_match.loc[~mask, 'has_deped_school_id'] = True

df_match.reset_index(inplace=True)
print(df_match.shape)

(4550, 7)


In [16]:
df_match['has_deped_school_id'].sum()

3621

Note: There are ESC schools that did **not** exist yet in SY 2023-2024 hence the False values in `has_deped_school_id`.

In [17]:
display(df_match)

,esc_school_id,count_addon_slots,count_fixed_slots,count_incentive_slots,total_count_slots,school_id,has_deped_school_id
0,303929,0,38,0,38,418013,True
1,1104375,0,50,0,50,408365,True
2,1504390,44,50,0,94,406245,True
3,304395,0,50,0,50,419005,True
4,1404408,0,50,0,50,nan,False
...,...,...,...,...,...,...,...
4545,1704000,0,0,0,0,nan,False
4546,1704001,0,12,0,12,408013,True
4547,1704002,10,50,0,60,403391,True
4548,1704003,0,11,0,11,407742,True


# 3.0. Unutilized slots
(Feb 4, 2026) ECAIR received data from GASS on official unutilized slots of ESC delivering schools in SY 2023-2024.

## 3.1. Load

In [18]:
fname = "SY 23-24 ESC Slots as of 020426.xlsx"
fpath = str(DATA_PRIVATE_DIR / fname)
esc_until = pd.read_excel(fpath, sheet_name='SUMMARY')
print(esc_until.shape)

(4640, 18)


In [20]:
display(esc_until.head(2))
display(esc_until.tail(2))

,School ID,School Name,Unnamed: 2,VALUE,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,School ID.1,School Name.1,Unnamed: 12,VALUE.1,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,NaN,NaN,FIXED,INCENTIVE,ADDITIONAL,TOTAL SLOT,BILLED,UNUTILIZED SLOTS,NaN,NaN,NaN,NaN,FIXED,INCENTIVE,ADDITIONAL,TOTAL SLOT,BILLED,UNUTILIZED SLOTS
1,303929.0,"One La Salle Educational Foundation, Inc.",38,0,0,38,32,6,NaN,NaN,903459,"Sibugay Technical Institute, Inc.",191,0,0,191,0,191


,School ID,School Name,Unnamed: 2,VALUE,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,School ID.1,School Name.1,Unnamed: 12,VALUE.1,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
4638,1704253.0,Southwestern Institute of Business and Technol...,50,0,0,50,39,11,NaN,NaN,701220,University of San Carlos (South Campus),202,0,15,217,310,-93
4639,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TOTAL,NaN,239956,0,52351,292307,226483,65824


## 3.2. Clean

In [43]:
tmp = esc_until.copy()
half_headers = [
    'esc_school_id','school_name',
    'slots_fixed','slots_incentive',
    'slots_additional','slots_total',
    'slots_billed','slots_unutilized',
]

half_idxs = [(0,8), (10,18)]

halves = []
for i, tup_idx in enumerate(half_idxs):
    half_a = tmp.iloc[1:,tup_idx[0]:tup_idx[1]]
    half_a.columns = half_headers
    
    # Drop rows whose school id is Nan
    half_a = half_a.dropna(subset='esc_school_id')
    
    # Establish data types
    for col in ['esc_school_id','school_name']:
        half_a[col] = half_a[col].astype(str)
    
    for col in half_a.loc[:,"slots_fixed":].columns:
        half_a[col] = half_a[col].astype(int)

    # Only the first half has school IDs that have a decimal
    if i == 0:
        # Clean esc school ids
        pattern = r"(.*)\.0"
        half_a['esc_school_id'] = half_a['esc_school_id'].str.extract(pattern)
    elif i == 1:
        half_a = half_a.iloc[:-1,:]  # Remove TOTAL row
    
    # display(half_a.head())
    halves.append(half_a)
    # break

halves = pd.concat(halves, axis=0, ignore_index=True)
print(halves.shape)

# I don't trust slots_unutilized so I will recompute
halves['slots_total_minus_billed'] = halves['slots_total'] - halves['slots_billed']

(9276, 8)


## 3.3. Inspect

In [46]:
display(halves.head())

,esc_school_id,school_name,slots_fixed,slots_incentive,slots_additional,slots_total,slots_billed,slots_unutilized,slots_total_minus_billed
0,303929,"One La Salle Educational Foundation, Inc.",38,0,0,38,32,6,6
1,1104375,Newfoundland Arts and Science Academy of Tagum...,50,0,0,50,23,27,27
2,1504390,4th Watch Maranatha Christian Academy of Bagui...,50,0,44,94,27,67,67
3,304395,ADVENT SCHOOL FOUNDATION INC.,50,0,0,50,20,30,30
4,304350,"ASKI Skills and Knowledge Institute, Inc.",50,0,0,50,50,0,0


In [45]:
# Check if I just have trust issues
display(halves[halves['slots_total_minus_billed'] != halves['slots_unutilized']])

,esc_school_id,school_name,slots_fixed,slots_incentive,slots_additional,slots_total,slots_billed,slots_unutilized,slots_total_minus_billed


## 3.4. DepEd IDs

In [68]:
# We (again) use our ESC Beneficiary datasets to get ESC-DepEd ID pairs
fname = "processed_esc_beneficiaries.parquet"
fpath = str(OUTPUT_DIR / fname)
benef = pd.read_parquet(fpath, engine='fastparquet')
# print(benef.shape)

# Process
benef_ids = benef[['esc_school_id', 'destination_school_id']].copy()

# Drop duplicating esc_school_id to make a table with unique esc schools
benef_ids = benef_ids.drop_duplicates(subset='esc_school_id')

for col in benef_ids.columns:
    benef_ids[col] = benef_ids[col].astype(str)

print(benef_ids.shape)

# Make a lookup dictionary
esc_deped_dict = benef_ids.set_index('esc_school_id').to_dict()

# Remove very heavy benef object from memory
del benef

(3767, 2)


In [52]:
display(benef_ids.head(2))

,esc_school_id,destination_school_id
0,1603520,475511
611,1202204,405799


In [55]:
esc_deped_dict.keys()

dict_keys(['destination_school_id'])

In [72]:
mrg_halves = halves.copy()

mrg_halves['school_id'] = mrg_halves['esc_school_id'].map(esc_deped_dict['destination_school_id'])
# print(mrg_halves.shape)

# Tag esc school IDs with no DepEd ID counterpart
mrg_halves['has_deped_school_id'] = True
mask_na = mrg_halves['school_id'].isna()
mrg_halves.loc[mask_na, 'has_deped_school_id'] = False

# Rearrange columns
ord_cols = [
    'school_id', 'esc_school_id', 'school_name',
    'slots_total', 'slots_unutilized',
    'has_deped_school_id'
]
mrg_halves = mrg_halves[ord_cols]

print(mrg_halves.shape)

(9276, 6)


In [73]:
display(mrg_halves.isna().sum())

school_id              1742
esc_school_id             0
school_name               0
slots_total               0
slots_unutilized          0
has_deped_school_id       0
dtype: int64

### Inspect no IDs

In [74]:
display(
    mrg_halves[mrg_halves['school_id'].isna()]
)

,school_id,esc_school_id,school_name,slots_total,slots_unutilized,has_deped_school_id
14,NaN,304358,Discovery Child Development of Montessori Inc.,50,50,False
18,NaN,1404397,"Ingenium School, Inc",50,50,False
48,NaN,1404383,"Southville International School and Colleges, ...",50,41,False
68,NaN,1404387,"Philippine Cultural College, Inc",50,50,False
69,NaN,1404417,"Philippine Cultural College, Inc",50,50,False
...,...,...,...,...,...,...
8752,NaN,1702400,Center for Positive Futures,0,0,False
8753,NaN,1702403,Puerto Princesa Adventist Elementary School,0,0,False
8755,NaN,1702622,Fullbright College Integrated School,0,0,False
8756,NaN,1703707,Southfields Educational Foundation Philippines...,0,0,False


# 4.0. Save
Previously, we save the output of section 2. Now, as of Feb. 4, 2026, we save the output of Section 3.

In [75]:
fpath = str(OUTPUT_DIR / 'processed_esc_slots.parquet')
mrg_halves.to_parquet(fpath)
print(f"Saving successful!")

Saving successful!
